# Customer Churn — Exploratory Data Analysis

This notebook analyzes only the 7,043 observed IBM Telco Customer Churn snapshots. It filters the scored export to `record_source == 'observed'`; the 21,000-row analytics scenario is excluded from every notebook calculation.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import load_config, resolve_path

sns.set_theme(style="whitegrid")
config = load_config()
scored_df = pd.read_csv(resolve_path(config["paths"]["scored_data"]))
if 'record_source' not in scored_df.columns:
    raise KeyError("Expected 'record_source' in the scored export. Run the current pipeline first.")
df = scored_df.loc[scored_df['record_source'].eq('observed')].copy()
print(f"Observed records: {len(df):,}")
print(f"Observed churn rate: {df['Churn'].mean():.1%}")
df.head()

## 1. Overall Churn Rate

In [ ]:
churn_counts = df['Churn'].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
churn_counts.plot(kind='bar', ax=ax, color=['#059669', '#dc2626'])
ax.set_title(f"Overall Observed Churn Rate: {df['Churn'].mean():.1%}")
ax.set_xticklabels(['Retained', 'Churned'], rotation=0)
plt.show()

## 2. Tenure Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.kdeplot(data=df, x='tenure', hue=df['Churn'].map({1: 'Churned', 0: 'Retained'}), fill=True, ax=ax)
ax.set_title('Tenure Distribution by Churn Status')
plt.show()

## 3. Contract Type & Payment Method

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby('Contract')['Churn'].mean().sort_values(ascending=False).plot(kind='bar', ax=axes[0], color='#2563eb')
axes[0].set_title('Observed Churn Rate by Contract')
df.groupby('PaymentMethod')['Churn'].mean().sort_values(ascending=False).plot(kind='barh', ax=axes[1], color='#7c3aed')
axes[1].set_title('Observed Churn Rate by Payment Method')
plt.tight_layout()
plt.show()

## 4. Internet Service & Estimated Support-Need Proxy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby('InternetService')['Churn'].mean().plot(kind='bar', ax=axes[0], color='#0891b2')
axes[0].set_title('Observed Churn by Internet Service')
sns.boxplot(data=df, x='support_ticket_proxy', y=df['Churn'].map({1: 'Churned', 0: 'Retained'}), orient='h', ax=axes[1])
axes[1].set_title('Estimated Support-Need Proxy by Churn Status')
plt.tight_layout()
plt.show()

## 5. Feature Correlations

In [ ]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'support_ticket_proxy',
                'contract_risk_score', 'customer_engagement_score', 'Churn']
corr = df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.show()

## Key Findings

1. **Month-to-month contracts** have the highest observed churn and warrant a retention-segment review.
2. **Fiber optic subscribers** show elevated observed churn; this is an investigation signal, not a causal conclusion.
3. **New customers (≤6 months tenure)** are the highest-risk lifecycle segment in this snapshot.
4. **Electronic check payers** and the estimated support-need proxy correlate with observed churn.
5. **Engagement score** and security add-ons are useful segmentation features.

The support measure is an engineered proxy, not observed support-ticket data; the dataset is cross-sectional.